# NVE snow and hydrology ? Bronze ingestion
Daily modelled grid-cell data from NVE GridTimeSeries (no API key). Run in Databricks with a Unity Catalog-enabled Spark/Delta runtime and `requests` installed. Keep `nve_snow_hydrology.py` beside this notebook (import both through a Databricks Git folder).

Output: `nve_snow_hydrology` observations and `nve_snow_hydrology_raw` responses in the configured catalog/schema. Raw records retain source JSON, theme metadata and request provenance. Reruns update matching observations and requests. Each successful request commits separately; failed jobs can be rerun safely.

These are point samples, not bidding-zone averages or observed reservoir inflow. The included coordinate is a demonstration point only. Configure your own catchment/grid points before analysis. NVE source wall-clock timestamps are retained without assigning a timezone.


In [ ]:
import json
import re
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, DateType,
)
from nve_snow_hydrology import (
    BASE_URL, DEFAULT_THEMES, create_session, date_windows, get_json,
    normalize, select_themes, validate_locations,
)


## Configuration
Dates are inclusive. Defaults refresh the preceding 30 days, ending yesterday in Europe/Oslo. Set `start_date` and `end_date` for a historical backfill. Schedule daily with the rolling defaults to capture recent revisions; rerun older dates to refresh older revisions.

Coordinates must be integer metres in UTM zone 33N (EPSG:25833). The example matches a verified NVE response at 1,226 m altitude and has no assigned power-market area. Use reviewed locations for production.


In [ ]:
today = datetime.now(ZoneInfo("Europe/Oslo")).date()
defaults = {
    "catalog": "dbw_dariusz",
    "schema": "bronze",
    "start_date": (today - timedelta(days=30)).isoformat(),
    "end_date": (today - timedelta(days=1)).isoformat(),
    "window_days": "366",
    "themes": ",".join(DEFAULT_THEMES),
    "locations_json": json.dumps([
        {"location_id": "example_utm33_130000_6800000", "x": 130000, "y": 6800000},
    ]),
}
for name, default in defaults.items():
    dbutils.widgets.text(name, default)
config = {name: dbutils.widgets.get(name).strip() for name in defaults}
for name in ("catalog", "schema"):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", config[name]):
        raise ValueError(f"Invalid {name}")
namespace = f"{config['catalog']}.{config['schema']}"
locations = json.loads(config["locations_json"])
validate_locations(locations)
themes = [theme.strip() for theme in config["themes"].split(",")]
window_days = int(config["window_days"])
if not 1 <= window_days <= 366:
    raise ValueError("window_days must be between 1 and 366")
windows = list(date_windows(config["start_date"], config["end_date"], window_days))
if datetime.fromisoformat(config["end_date"]).date() >= today:
    raise ValueError("Use an end_date before today; this job ingests historical daily data")


## Explicit Bronze schemas and idempotent writes
Observation key: `(x, y, theme, source_time, time_resolution_minutes)`. Location IDs are descriptive labels; coordinates identify the point. Missing-value sentinels become null with `is_missing=true`; zero remains zero. Units come from the standard endpoint response, which already converts temperature to Celsius and snow depth to centimetres. No additional scaling is applied.

Raw-response key: `(x, y, theme, requested_start, requested_end)`. It stores the latest response for each request, rather than a version history. Neither table is overwritten. The job requires create-table and merge privileges; run one ingestion job at a time for these tables.


In [ ]:
observation_schema = StructType([
    StructField("location_id", StringType(), False),
    StructField("x", IntegerType(), False), StructField("y", IntegerType(), False),
    StructField("theme", StringType(), False), StructField("theme_name", StringType(), True),
    StructField("source_time", StringType(), False),
    StructField("observation_date", DateType(), False),
    StructField("time_resolution_minutes", IntegerType(), False),
    StructField("value", DoubleType(), True), StructField("is_missing", BooleanType(), False),
    StructField("no_data_value", DoubleType(), True), StructField("unit", StringType(), True),
    StructField("altitude_m", DoubleType(), True),
])
raw_schema = StructType([
    StructField("location_id", StringType(), False),
    StructField("x", IntegerType(), False), StructField("y", IntegerType(), False),
    StructField("theme", StringType(), False),
    StructField("requested_start", StringType(), False),
    StructField("requested_end", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("payload_json", StringType(), False),
    StructField("theme_metadata_json", StringType(), False),
])

def merge_rows(rows, schema, table, keys):
    if not rows:
        raise ValueError(f"Refusing empty batch for {table}")
    frame = (spark.createDataFrame(rows, schema)
             .withColumn("ingested_at", F.current_timestamp())
             .withColumn("source", F.lit("NVE_GRIDTIMESERIES")))
    qualified = f"{namespace}.{table}"
    if not spark.catalog.tableExists(qualified):
        frame.write.format("delta").mode("errorifexists").saveAsTable(qualified)
    else:
        condition = " AND ".join(f"target.`{key}` = incoming.`{key}`" for key in keys)
        (DeltaTable.forName(spark, qualified).alias("target")
         .merge(frame.alias("incoming"), condition)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


## Fetch, validate, and merge
The theme catalogue is validated before writes. Requests use connect/read timeouts and bounded retries for throttling/transient server errors. Response coordinates, dates, cadence, and array length must match before a batch is written. Failures stop the job; earlier committed batches remain available. Raw data is committed before observations so a failed observation write can be recovered by rerunning the same dates.


In [ ]:
completed_requests = 0
observation_count = 0
missing_count = 0
with create_session() as session:
    metadata = get_json(session, "GridTimeSeries/Themes/json")
    selected_metadata = select_themes(metadata, themes, config["start_date"])
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {namespace}")
    for location in locations:
        for theme in themes:
            for start, end in windows:
                path = f"GridTimeSeries/{location['x']}/{location['y']}/{start}/{end}/{theme}.json"
                payload = get_json(session, path)
                rows = normalize(payload, location, theme, start, end)
                raw_row = {
                    **location, "theme": theme, "requested_start": start, "requested_end": end,
                    "source_url": f"{BASE_URL}/{path}",
                    "payload_json": json.dumps(payload, ensure_ascii=False, allow_nan=False),
                    "theme_metadata_json": json.dumps(selected_metadata[theme], ensure_ascii=False),
                }
                # Keep only schema fields if location configuration contains extra descriptive attributes.
                raw_row = {field.name: raw_row[field.name] for field in raw_schema}
                merge_rows([raw_row], raw_schema, "nve_snow_hydrology_raw",
                           ["x", "y", "theme", "requested_start", "requested_end"])
                merge_rows(rows, observation_schema, "nve_snow_hydrology",
                           ["x", "y", "theme", "source_time", "time_resolution_minutes"])
                completed_requests += 1
                observation_count += len(rows)
                missing_count += sum(row["is_missing"] for row in rows)
                print(f"{location['location_id']} / {theme} / {start}..{end}: {len(rows)} rows")
print(f"Completed {completed_requests} requests; upserted {observation_count} observations "
      f"({missing_count} missing values). Target: {namespace}.nve_snow_hydrology")


## Source and interpretation
- [NVE GridTimeSeries documentation](https://api.nve.no/doc/gridtimeseries-data-gts/)
- [Live theme catalogue](https://gts.nve.no/api/GridTimeSeries/Themes/json)
- `swe`: snow water equivalent (mm); `sd`: snow depth (cm).
- `qtt`: rain plus snowmelt (mm); `gwb_q`: modelled runoff (mm).
- `rr`: precipitation (mm); `tm`: temperature (Celsius).

NVE data is licensed under NLOD, compatible with CC BY 3.0 Norway. Attribute NVE and link to the service in downstream outputs. Grid modelled runoff in mm is not measured river discharge in m?/s. For observed station data use [NVE HydAPI](https://hydapi.nve.no/UserDocumentation/), which requires a separately registered API key; this notebook does not fetch those observations.
